# Ensemble methods on Titanic 🚢🚢

## Introduction

This exercise is the opportunity to practice ensemble methods on a dataset you have worked with before and that is the Titanic dataset.

Let's start by importing the librairies that we will used in the exercise.

In [1]:
# Load in our libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer, KNNImputer
# import ensemble methods
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

## Feature Exploration, Engineering and Cleaning 

1. Import the data using the following link : "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/stacking/titanic.csv" , and display the first lines. Are there any missing values in the dataset?

In [2]:
dataset = pd.read_csv("https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+Supervis%C3%A9/stacking/titanic.csv")
dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
dataset.describe(include='all')

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,B96 B98,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [4]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


2. What types of variables are present in this dataset? What kind of preprocessing could you run on these variables?

3. Here are some guidelines you can follow to clean the dataset as well as create new variables (feature engineering).

a.  Create a Name_length variable that measures the number of characters in the variable Name for each observations.

In [5]:
dataset['Name_length'] = dataset['Name'].apply(lambda x: len(x))
dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24


b. Create a variable Has_Cabin that indicates whether the passenger has a cabin or not.

Hint: [this method](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.notna.html#pandas.DataFrame.notna) might be useful 😉

In [6]:
dataset['Has_cabin'] = dataset['Cabin'].apply(lambda x: False if pd.isna(x) else True)
dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length,Has_cabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False


c. Create a variable FamilySize that gives the size of each passenger's family.

In [7]:
dataset['Family_size'] = dataset['SibSp'] + dataset['Parch'] + 1
dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length,Has_cabin,Family_size
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False,2
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True,2
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False,1
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True,2
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False,1


d. Create a variable IsAlone that indicates whether the passenger is traveling on their own.

In [8]:
dataset['Is_alone'] = dataset['Family_size'].apply(lambda x: True if x == 1 else False)
dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length,Has_cabin,Family_size,Is_alone
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False,2,False
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True,2,False
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False,1,True
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True,2,False
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False,1,True


h. Extract the title from each passenger in order to create a variable Title.

Hint: You might consider _applying_ a function that calls the [str.split method](https://docs.python.org/3.3/library/stdtypes.html?highlight=split#str.split) 😉

In [9]:
dataset['Title'] = dataset['Name'].apply(lambda x: x.split(',')[1].split('.')[0])
dataset.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Name_length,Has_cabin,Family_size,Is_alone,Title
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,23,False,2,False,Mr
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,51,True,2,False,Mrs
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,22,False,1,True,Miss
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,44,True,2,False,Mrs
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,24,False,1,True,Mr


i. If some of these titles are equivalent convert them in order to bring them all in the same category.

In [10]:
data_title = dataset['Title'].value_counts().reset_index()
data_title



,Title,count
0,Mr,517
1,Miss,182
2,Mrs,125
3,Master,40
4,Dr,7
5,Rev,6
6,Mlle,2
7,Major,2
8,Col,2
9,the Countess,1


In [11]:
fig  = px.bar(
    data_title,
    x = 'Title',
    y = 'count',
    title = 'Distribution of Titles in the Titanic Dataset',
    labels = {'index': 'Title', 'Title': 'Count'},
)
fig.show()

j. Are any of the remaining titles underrepresented among the observations? If it is the case, group them in a unique modality "Rare"

In [12]:
rare_titles = data_title[data_title['count'] < 10]['Title'].tolist()
rare_titles

[' Dr',
 ' Rev',
 ' Mlle',
 ' Major',
 ' Col',
 ' the Countess',
 ' Capt',
 ' Ms',
 ' Sir',
 ' Lady',
 ' Mme',
 ' Don',
 ' Jonkheer']

In [13]:
data_title['Title'] = data_title['Title'].apply(lambda x: 'Rare' if x in rare_titles else x)
data_title = data_title.groupby('Title').sum().sort_values(by='count', ascending=False).reset_index()
data_title

,Title,count
0,Mr,517
1,Miss,182
2,Mrs,125
3,Master,40
4,Rare,27


In [14]:
fig = px.bar(
    data_title,
    x = 'Title',
    y = 'count',
    title = 'Distribution of Titles in the Titanic Dataset',
    labels = {'index': 'Title', 'Title': 'Count'},
)
fig.show()

4. Drop the columns 'PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp' du dataset. Why don't we need these columns for what's next?

In [15]:
dataset = dataset.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp'])

In [33]:
dataset['Title'] = dataset['Title'].apply(lambda x: 'Rare' if x in rare_titles else x)
dataset.sample(10)

,Survived,Pclass,Sex,Age,Parch,Fare,Embarked,Name_length,Has_cabin,Family_size,Is_alone,Title
565,0,3,male,24.0,0,24.1500,S,20,False,3,False,Mr
740,1,1,male,NaN,0,30.0000,S,27,True,1,True,Mr
245,0,1,male,44.0,0,90.0000,Q,27,True,3,False,Rare
236,0,2,male,44.0,0,26.0000,S,17,False,2,False,Mr
291,1,1,female,19.0,0,91.0792,C,39,True,2,False,Mrs
393,1,1,female,23.0,0,113.2750,C,22,True,2,False,Miss
555,0,1,male,62.0,0,26.5500,S,18,False,1,True,Mr
395,0,3,male,22.0,0,7.7958,S,19,False,1,True,Mr
754,1,2,female,48.0,2,65.0000,S,32,False,4,False,Mrs
369,1,1,female,24.0,0,69.3000,C,29,True,1,True,Rare


In [34]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Survived     891 non-null    int64  
 1   Pclass       891 non-null    int64  
 2   Sex          891 non-null    object 
 3   Age          714 non-null    float64
 4   Parch        891 non-null    int64  
 5   Fare         891 non-null    float64
 6   Embarked     889 non-null    object 
 7   Name_length  891 non-null    int64  
 8   Has_cabin    891 non-null    bool   
 9   Family_size  891 non-null    int64  
 10  Is_alone     891 non-null    bool   
 11  Title        891 non-null    object 
dtypes: bool(2), float64(2), int64(5), object(3)
memory usage: 71.5+ KB


5. Separate the features from the target and split the data between train and test (with random_state = 0)

In [39]:
target = 'Survived'

X = dataset.loc[:, dataset.columns != target]
Y = dataset[target]

print(X.head())
print(Y.head())

X_train, X_test, y_train, y_test = train_test_split(X, Y, stratify = Y, random_state = 0)

   Pclass     Sex   Age  Parch     Fare Embarked  Name_length  Has_cabin  \
0       3    male  22.0      0   7.2500        S           23      False   
1       1  female  38.0      0  71.2833        C           51       True   
2       3  female  26.0      0   7.9250        S           22      False   
3       1  female  35.0      0  53.1000        S           44       True   
4       3    male  35.0      0   8.0500        S           24      False   

   Family_size  Is_alone  Title  
0            2     False     Mr  
1            2     False    Mrs  
2            1      True   Miss  
3            2     False    Mrs  
4            1      True     Mr  
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


6. Using the Pipeline and ColumnTransformer, make all the preprocessings at once. Use the [KNN imputer](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html) to handle the missing values in the numeric variables, and the SimpleImputer for categorical data.

In [40]:
X.dtypes

Pclass           int64
Sex             object
Age            float64
Parch            int64
Fare           float64
Embarked        object
Name_length      int64
Has_cabin         bool
Family_size      int64
Is_alone          bool
Title           object
dtype: object

In [41]:
num_features = []
cat_features = []

for col in X.columns:
    if X[col].dtype in ['int64', 'float64', 'bolean']:
        num_features.append(col)
    else:
        cat_features.append(col)

print("Numerical features:", num_features)
print("Categorical features:", cat_features)

Numerical features: ['Pclass', 'Age', 'Parch', 'Fare', 'Name_length', 'Family_size']
Categorical features: ['Sex', 'Embarked', 'Has_cabin', 'Is_alone', 'Title']


In [42]:
numeric_transformer = Pipeline(steps=[
    ('imputer', KNNImputer()),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_features),
        ('cat', categorical_transformer, cat_features)
    ])

X_train = preprocessor.fit_transform(X_train)
print(X_train[:,5])
X_test = preprocessor.transform(X_test)
print(X_test[:,5])

[ 0.08547193  0.72699155  2.65155041  0.72699155 -0.5560477  -0.5560477
  0.08547193 -0.5560477  -0.5560477   0.08547193 -0.5560477  -0.5560477
 -0.5560477   0.72699155 -0.5560477  -0.5560477  -0.5560477   1.36851117
  2.65155041  0.08547193  0.08547193 -0.5560477  -0.5560477   0.08547193
 -0.5560477   5.85914852 -0.5560477  -0.5560477   0.72699155  0.72699155
 -0.5560477   0.08547193  0.08547193 -0.5560477  -0.5560477   0.72699155
  2.01003079 -0.5560477  -0.5560477  -0.5560477   0.08547193  0.08547193
  0.08547193 -0.5560477   2.01003079 -0.5560477   0.72699155 -0.5560477
 -0.5560477  -0.5560477  -0.5560477   0.08547193 -0.5560477  -0.5560477
 -0.5560477   2.65155041  0.08547193 -0.5560477   0.08547193 -0.5560477
 -0.5560477  -0.5560477  -0.5560477  -0.5560477   1.36851117 -0.5560477
 -0.5560477  -0.5560477  -0.5560477  -0.5560477  -0.5560477  -0.5560477
  2.65155041  0.08547193 -0.5560477   5.85914852 -0.5560477  -0.5560477
 -0.5560477   2.01003079 -0.5560477  -0.5560477   0.7269915

### Pearson Correlation Heatmap

7. Produce a figure that contains the correlation table for all the explanatory variables of X_train, what do you think?

In [44]:
corr_matrix = pd.DataFrame(X_train).corr().round(2)
corr_matrix

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,1.00,-0.37,-0.00,-0.55,-0.21,0.05,0.15,0.22,0.09,-0.74,0.16,-0.02,0.15,-0.16,-0.20
1,-0.37,1.00,-0.19,0.09,0.02,-0.26,0.05,-0.06,-0.01,0.26,0.19,-0.22,0.18,0.17,0.13
2,-0.00,-0.19,1.00,0.20,0.25,0.79,-0.23,-0.06,0.02,0.05,-0.58,0.09,-0.33,0.22,-0.06
3,-0.55,0.09,0.20,1.00,0.15,0.20,-0.17,-0.11,-0.20,0.48,-0.26,0.13,-0.17,0.08,0.02
4,-0.21,0.02,0.25,0.15,1.00,0.25,-0.44,-0.11,0.08,0.17,-0.43,-0.05,-0.44,0.63,0.02
5,0.05,-0.26,0.79,0.20,0.25,1.00,-0.18,-0.03,0.04,0.01,-0.70,0.08,-0.34,0.16,-0.06
6,0.15,0.05,-0.23,-0.17,-0.44,-0.18,1.00,-0.09,0.10,-0.14,0.28,-0.68,0.86,-0.56,0.04
7,0.22,-0.06,-0.06,-0.11,-0.11,-0.03,-0.09,1.00,-0.50,-0.14,0.07,0.18,-0.11,-0.08,0.01
8,0.09,-0.01,0.02,-0.20,0.08,0.04,0.10,-0.50,1.00,-0.10,0.06,-0.13,0.10,0.02,-0.03
9,-0.74,0.26,0.05,0.48,0.17,0.01,-0.14,-0.14,-0.10,1.00,-0.18,0.05,-0.12,0.11,0.07


In [ ]:
import plotly.figure_factory as ff

fig = ff.create_annotated_heatmap(corr_matrix.values,
                                x = corr_matrix.columns.tolist(),
                                y = corr_matrix.index.tolist())


fig.show()

**Correlations between the variables are not very high, we can hope that they will each bring complementary information in order to predict the target variable.**

## Ensembling & Stacking models

Now that we have finished our preprocessing and made sure our data was fit for prediction, let's move on to creating our ensemble models. We'll train different models with different ensembling strategies and store their train and test scores for comparison.

### Random Forest
8. Train a Random Forest by tuning the hyperparameters with a grid search. Which ensemble method is related to random forests?

Evaluate the best model's accuracy on train and test sets. Save the scores into a pandas DataFrame.

In [46]:
scores_df = pd.DataFrame(columns = ['model', 'accuracy', 'set'])

In [47]:
# Perform grid search
print("Grid search...")
random_forest = RandomForestClassifier()

# Grid of values to be tested
params = {
    'max_depth': [2, 4, 6, 8, 10],
    'min_samples_leaf': [1, 2, 5],
    'min_samples_split': [2, 4, 8],
    'n_estimators': [10, 20, 40, 60, 80, 100]
}
print(params)
gridsearch = GridSearchCV(random_forest, param_grid = params, cv = 3, verbose = 1) # cv : the number of folds to be used for CV
gridsearch.fit(X_train, y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, y_test))


# scores_df = scores_df.append({'model': 'random_forest', 'accuracy': gridsearch.score(X_train, y_train), 'set': 'train'}, ignore_index = True)
# scores_df = scores_df.append({'model': 'random_forest', 'accuracy': gridsearch.score(X_test, y_test), 'set': 'test'}, ignore_index = True)
# scores_df

new_rows = [
    {'model': 'random_forest', 'accuracy': gridsearch.score(X_train, y_train), 'set': 'train'},
    {'model': 'random_forest', 'accuracy': gridsearch.score(X_test, y_test), 'set': 'test'}
]

scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df


Grid search...
{'max_depth': [2, 4, 6, 8, 10], 'min_samples_leaf': [1, 2, 5], 'min_samples_split': [2, 4, 8], 'n_estimators': [10, 20, 40, 60, 80, 100]}
Fitting 3 folds for each of 270 candidates, totalling 810 fits
...Done.
Best hyperparameters :  {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 4, 'n_estimators': 20}
Best validation accuracy :  0.8278053838591956

Accuracy on training set :  0.8787425149700598
Accuracy on test set :  0.8161434977578476


,model,accuracy,set
0,random_forest,0.878743,train
1,random_forest,0.816143,test


9. Create your own Bagging of decision tree (with the same hyperparameters as the optimal ones for Random Forest) and check you get compatible performances.

In [49]:
decision_tree = DecisionTreeClassifier(max_depth = 8, min_samples_leaf = 1, min_samples_split = 4)
bagging = BaggingClassifier(estimator=decision_tree, n_estimators=100, random_state=0)
bagging.fit(X_train, y_train)

print("Bagging accuracy on training set : ", bagging.score(X_train, y_train))
print("Bagging accuracy on test set : ", bagging.score(X_test, y_test))

new_rows = [
    {'model': 'bagging', 'accuracy': bagging.score(X_train, y_train), 'set': 'train'},
    {'model': 'bagging', 'accuracy': bagging.score(X_test, y_test), 'set': 'test'}
]

scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df


Bagging accuracy on training set :  0.9446107784431138
Bagging accuracy on test set :  0.8116591928251121


,model,accuracy,set
0,random_forest,0.878743,train
1,random_forest,0.816143,test
2,bagging,0.944611,train
3,bagging,0.811659,test


10. Train an AdaBoost model by tuning the hyperparameters:
* With a logistic regression as base estimator
* With a decision tree as base estimator

For each model, evaluate the performances on the test set.

In [51]:
logistic_regression = LogisticRegression(max_iter=1000)
adaboost_logreg = AdaBoostClassifier(logistic_regression)

adaboost_logreg.fit(X_train, y_train)

params = {
    'estimator__C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0],
    'n_estimators': [5, 10, 20, 40, 60, 80, 100]
}

grid_search = GridSearchCV(adaboost_logreg, param_grid=params, cv=3, verbose=1)
grid_search.fit(X_train, y_train)

new_rows = [
    {'model': 'adaboost_logreg', 'accuracy': grid_search.score(X_train, y_train), 'set': 'train'},
    {'model': 'adaboost_logreg', 'accuracy': grid_search.score(X_test, y_test), 'set': 'test'}
]   
scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df

Fitting 3 folds for each of 56 candidates, totalling 168 fits


,model,accuracy,set
0,random_forest,0.878743,train
1,random_forest,0.816143,test
2,bagging,0.944611,train
3,bagging,0.811659,test
4,adaboost_logreg,0.829341,train
5,adaboost_logreg,0.816143,test


In [52]:
decision_tree = DecisionTreeClassifier()
adaboost_dt = AdaBoostClassifier(decision_tree)
adaboost_dt.fit(X_train, y_train)

params = {
    'estimator__max_depth': [8, 10, 12],
    'estimator__min_samples_leaf': [1, 2, 3],
    'estimator__min_samples_split': [6, 8, 10],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}

grid_search = GridSearchCV(adaboost_dt, param_grid=params, cv=3, verbose=1)
grid_search.fit(X_train, y_train)

new_rows = [
    {'model': 'adaboost_dt', 'accuracy': grid_search.score(X_train, y_train), 'set': 'train'},
    {'model': 'adaboost_dt', 'accuracy': grid_search.score(X_test, y_test), 'set': 'test'}
]
scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df

Fitting 3 folds for each of 162 candidates, totalling 486 fits


,model,accuracy,set
0,random_forest,0.878743,train
1,random_forest,0.816143,test
2,bagging,0.944611,train
3,bagging,0.811659,test
4,adaboost_logreg,0.829341,train
5,adaboost_logreg,0.816143,test
6,adaboost_dt,1.000000,train
7,adaboost_dt,0.820628,test


11. Train scikit-learn's GradientBoosting model (by tuning hyperparameters) and evaluate the performances.

In [53]:
gradient_boosting = GradientBoostingClassifier()

params = {
    'max_depth': [8, 10, 12],
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [6, 8, 10],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}

grid_search = GridSearchCV(gradient_boosting, param_grid=params, cv=3, verbose=1)
grid_search.fit(X_train, y_train)

new_rows = [
    {'model': 'gradient_boosting', 'accuracy': grid_search.score(X_train, y_train), 'set': 'train'},
    {'model': 'gradient_boosting', 'accuracy': grid_search.score(X_test, y_test), 'set': 'test'}
]
scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df


Fitting 3 folds for each of 162 candidates, totalling 486 fits


,model,accuracy,set
0,random_forest,0.878743,train
1,random_forest,0.816143,test
2,bagging,0.944611,train
3,bagging,0.811659,test
4,adaboost_logreg,0.829341,train
5,adaboost_logreg,0.816143,test
6,adaboost_dt,1.000000,train
7,adaboost_dt,0.820628,test
8,gradient_boosting,0.932635,train
9,gradient_boosting,0.829596,test


12. Train an XGBoost model (by tuning hyperparameters). Do you get better or similar results compared to scikit-learn's GradientBoosting?

In [54]:
XGboost = XGBClassifier()

params = {
    'max_depth': [4, 6, 8, 10],
    'min_child_weight': [1, 2, 4, 6, 8],
    'n_estimators': [2, 4, 6, 8, 10, 12]
}

grid_search = GridSearchCV(XGboost, param_grid=params, cv=3, verbose=1)
grid_search.fit(X_train, y_train)

new_rows = [
    {'model': 'XGboost', 'accuracy': grid_search.score(X_train, y_train), 'set': 'train'},
    {'model': 'XGboost', 'accuracy': grid_search.score(X_test, y_test), 'set': 'test'}
]
scores_df = pd.concat([scores_df, pd.DataFrame(new_rows)], ignore_index=True)
scores_df

Fitting 3 folds for each of 120 candidates, totalling 360 fits


,model,accuracy,set
0,random_forest,0.878743,train
1,random_forest,0.816143,test
2,bagging,0.944611,train
3,bagging,0.811659,test
4,adaboost_logreg,0.829341,train
5,adaboost_logreg,0.816143,test
6,adaboost_dt,1.000000,train
7,adaboost_dt,0.820628,test
8,gradient_boosting,0.932635,train
9,gradient_boosting,0.829596,test


13. Compare all the models' performances in a bar chart and conclude. Which model is the best?

Hint: the option `barmode` in plotly's `px.bar()` might be useful 😇

In [59]:
scores_df = scores_df.sort_values(by = ['set', 'accuracy'], ascending = False)
scores_df

,model,accuracy,set
6,adaboost_dt,1.000000,train
2,bagging,0.944611,train
8,gradient_boosting,0.932635,train
10,XGboost,0.892216,train
0,random_forest,0.878743,train
4,adaboost_logreg,0.829341,train
9,gradient_boosting,0.829596,test
7,adaboost_dt,0.820628,test
1,random_forest,0.816143,test
5,adaboost_logreg,0.816143,test


In [60]:
fig = px.bar(
    scores_df,
    x='model',
    y='accuracy',
    barmode='group',
    color='set',
    title='Model Accuracy Comparison',
    labels={'model': 'Model', 'accuracy': 'Accuracy'}
)
fig.show()